# Fine-Tune a Model Router using Azure OpenAI REST API

This notebook walks you through end-to-end fine-tuning of a **Model Router** using Azure OpenAI REST endpoints. The workflow follows the official [Azure OpenAI fine-tuning guide](https://learn.microsoft.com/en-us/azure/foundry/openai/how-to/fine-tuning?tabs=oai-sdk&pivots=rest-api#review-the-workflow-for-the-rest-api).

## Workflow
1. **Configure** – Set all parameters in a single config cell.
2. **Understand the data format** – Review how training data is structured.
3. **Upload data** – Upload training (and optionally test) files to Azure OpenAI.
4. **Submit fine-tuning job** – Kick off model training.
5. **Monitor job** – Poll until training completes.
6. **Deploy the model** – Create a deployment for inference.
7. **Evaluate the deployment** – Evaluate the fine-tuned model.

## Prerequisites
- An Azure AI Foundry project with fine-tuning access.
- The **Azure AI Owner** role (required for deployment).
- Python packages: `requests`.
- An API key or Azure AD token for your resource.

## 0. Imports

All imports are consolidated here at the top of the notebook.

In [1]:
import json
import os
import sys
import requests
import subprocess

# Ensure the src/ helper module is importable
sys.path.insert(0, os.path.abspath("."))

from src.finetuning_helpers import (
    upload_file,
    create_finetuning_job,
    poll_job_until_complete,
    list_job_events,
    download_result_file,
    deploy_finetuned_model,
)

## 1. Configuration

Set all user-configurable parameters below. This is the **only cell** you need to edit before running the notebook.

In [ ]:
# ── Azure AI Foundry Project Endpoint ─────────────────────────────────────
#AZURE_OPENAI_ENDPOINT = "https://<YOUR_RESOURCE_NAME>.services.ai.azure.com/api/projects/<YOUR_PROJECT_NAME>"


AZURE_RESOURCE_NAME = "modelrouter-eus2"  # Replace with your actual resource name (without .openai.azure.com)
AZURE_PROJECT_NAME = "modelrouter-eus2-project"  
AZURE_OPENAI_ENDPOINT = f"https://{AZURE_RESOURCE_NAME}.services.ai.azure.com/api/projects/{AZURE_PROJECT_NAME}"  # Replace with your actual endpoint
AZURE_OPENAI_API_KEY  = "<YOUR_API_KEY>"  # Or set via environment variable

# ── Data Paths ─────────────────────────────────────────────────────────
TRAINING_FILE_PATH   = "data/contoso_clinic_train_v2_upload.jsonl"
VALIDATION_FILE_PATH = None #"data/zava_enterprise_test.jsonl"  # Optional – if not provided, training data is split 80:20 automatically

# ── Model ──────────────────────────────────────────────────────────────
BASE_MODEL = "model-router" 
SUFFIX     = "demo"      # Up to 18 chars; helps identify your fine-tuned model

# ── Deployment (used after training succeeds) ─────────────────────
SUBSCRIPTION_ID      = "72c03bf3-4e69-41af-9532-dfcdc3eefef4"
RESOURCE_GROUP       = "modelrouter-eus2-rg"
DEPLOYMENT_NAME      = "model-router-demo-contosoclinic"  # Name for the deployed model
DEPLOYMENT_CAPACITY  = 100

# ── Monitoring ─────────────────────────────────────────────────────────
POLL_INTERVAL_SECONDS = 60  # How often to check job status

## 2. Understanding the Data Format

The training data must be in **JSONL** (JSON Lines) format. Each line is a JSON object with the following structure:

```json
{
  "messages": [
    {"role": "user", "content": "<your prompt>"}
  ],
  "metrics": {
    "model_a": 1,
    "model_b": 0
  },
  "usage": {
    "model_a": {"completion_tokens": 205, "prompt_tokens": 184},
    "model_b": {"completion_tokens": 179, "prompt_tokens": 184}
  }
}
```

### Field descriptions

| Field | Required | Description |
|-------|----------|-------------|
| `messages` | ✅ Yes | A list of chat messages following the Chat Completions API format. Typically contains a `user` message with the prompt. |
| `metrics` | ✅ Yes | A dictionary mapping each model name to a binary score (`1` = correct, `0` = incorrect). The model router uses these to learn which models perform well on which types of tasks. |
| `usage` | ✅ Yes | A dictionary mapping each model name to its token usage (`completion_tokens` and `prompt_tokens`). The router uses this to optimize for cost-efficiency alongside accuracy. |

### Notes
- **Validation data is optional.** If you do not provide a validation file, the training data is automatically split **80:20** (80% train, 20% validation).
- Files must be UTF-8 encoded and less than 512 MB.
- A minimum of 200 training examples is recommended for meaningful results.

Let's preview a sample from the training data:

In [3]:
# Preview the first training example
with open(TRAINING_FILE_PATH, "r", encoding="utf-8") as f:
    sample = json.loads(f.readline())

print("=== Sample Training Record ===")
print(json.dumps(sample, indent=2)[:2000])  # Truncate for readability

=== Sample Training Record ===
{
  "messages": [
    {
      "role": "user",
      "content": "Can I have grapefruit when I'm on sertraline?"
    }
  ],
  "metrics": {
    "gpt-5_2025-08-07": 1,
    "gpt-5-mini_2025-08-07": 0,
    "gpt-5-nano_2025-08-07": 0
  },
  "usage": {
    "gpt-5_2025-08-07": {
      "prompt_tokens": 11,
      "completion_tokens": 40
    },
    "gpt-5-mini_2025-08-07": {
      "prompt_tokens": 11,
      "completion_tokens": 33
    },
    "gpt-5-nano_2025-08-07": {
      "prompt_tokens": 11,
      "completion_tokens": 26
    }
  }
}


## 3. Upload Data

Upload the training file (and optionally the validation file) to Azure OpenAI.

In [18]:
# Upload training file
print("Uploading training file...")
train_response = upload_file(AZURE_OPENAI_ENDPOINT, AZURE_OPENAI_API_KEY, TRAINING_FILE_PATH)
training_file_id = train_response["id"]
print(f"Training file ID: {training_file_id}")

# Upload validation file (optional)
validation_file_id = None
if VALIDATION_FILE_PATH:
    print("Uploading validation file...")
    val_response = upload_file(AZURE_OPENAI_ENDPOINT, AZURE_OPENAI_API_KEY, VALIDATION_FILE_PATH)
    validation_file_id = val_response["id"]
    print(f"Validation file ID: {validation_file_id}")
else:
    print("No validation file provided. Training data will be split 80:20 automatically.")

Uploading training file...
Training file ID: file-afcaa86c930e4388a917d3b342e3f3b1
No validation file provided. Training data will be split 80:20 automatically.


## 4. Submit Fine-Tuning Job

Create a fine-tuning job with the uploaded data. Hyperparameters are optional — the service uses sensible defaults if not specified.

In [19]:
print("Submitting fine-tuning job...")
job_response = create_finetuning_job(
    endpoint=AZURE_OPENAI_ENDPOINT,
    api_key=AZURE_OPENAI_API_KEY,
    model=BASE_MODEL,
    training_file_id=training_file_id,
    validation_file_id=validation_file_id,
    seed=105,  # For reproducibility
)

job_id = job_response["id"]
print(f"Fine-tuning job submitted!")
print(f"Job ID: {job_id}")
print(json.dumps(job_response, indent=2))

Submitting fine-tuning job...
Fine-tuning job submitted!
Job ID: ftjob-396b292fbfb940feb07f918603f0c010
{
  "hyperparameters": {
    "n_epochs": 1,
    "batch_size": 16,
    "learning_rate_multiplier": 1
  },
  "additionalParameters": {
    "completionOverride": "True"
  },
  "seed": 105,
  "method": {
    "type": "supervised",
    "supervised": {
      "hyperparameters": {
        "n_epochs": 1,
        "batch_size": 16,
        "learning_rate_multiplier": 1
      }
    }
  },
  "trainingType": "globalStandard",
  "status": "pending",
  "model": "model-router-2025-11-18",
  "metadata": {
    "base_model": "model-router",
    "model_version": "2025-11-18"
  },
  "training_file": "file-afcaa86c930e4388a917d3b342e3f3b1",
  "estimated_finish": 1779267820,
  "id": "ftjob-396b292fbfb940feb07f918603f0c010",
  "created_at": 1779264460,
  "object": "fine_tuning.job"
}


## 5. Monitor the Fine-Tuning Job

Poll the job status until it reaches a terminal state (`succeeded`, `failed`, or `cancelled`). This can take minutes to hours depending on dataset size and model.

In [20]:
# Poll until the job finishes
final_status = poll_job_until_complete(
    endpoint=AZURE_OPENAI_ENDPOINT,
    api_key=AZURE_OPENAI_API_KEY,
    job_id=job_id,
    poll_interval=POLL_INTERVAL_SECONDS,
)

print("\n=== Final Job Status ===")
print(json.dumps(final_status, indent=2))

Job ftjob-396b292fbfb940feb07f918603f0c010 status: pending
Job ftjob-396b292fbfb940feb07f918603f0c010 status: pending
Job ftjob-396b292fbfb940feb07f918603f0c010 status: pending
Job ftjob-396b292fbfb940feb07f918603f0c010 status: pending
Job ftjob-396b292fbfb940feb07f918603f0c010 status: pending
Job ftjob-396b292fbfb940feb07f918603f0c010 status: pending
Job ftjob-396b292fbfb940feb07f918603f0c010 status: running
Job ftjob-396b292fbfb940feb07f918603f0c010 status: running
Job ftjob-396b292fbfb940feb07f918603f0c010 status: running
Job ftjob-396b292fbfb940feb07f918603f0c010 status: running
Job ftjob-396b292fbfb940feb07f918603f0c010 status: running
Job ftjob-396b292fbfb940feb07f918603f0c010 status: running
Job ftjob-396b292fbfb940feb07f918603f0c010 status: running
Job ftjob-396b292fbfb940feb07f918603f0c010 status: running
Job ftjob-396b292fbfb940feb07f918603f0c010 status: running
Job ftjob-396b292fbfb940feb07f918603f0c010 status: running
Job ftjob-396b292fbfb940feb07f918603f0c010 status: runni

In [21]:
# View training events / logs
events = list_job_events(AZURE_OPENAI_ENDPOINT, AZURE_OPENAI_API_KEY, job_id)
print(json.dumps(events, indent=2))

{
  "has_more": false,
  "data": [
    {
      "id": "ftevent-eff97dd303fe4cbe88487ed6cbcef24f",
      "created_at": 1779265652,
      "level": "info",
      "message": "Training tokens billed: 0",
      "type": "message",
      "object": "fine_tuning.job.event"
    },
    {
      "id": "ftevent-55e0b7bec2fa4a06808c01ab89ecc5de",
      "created_at": 1779265652,
      "level": "info",
      "message": "Completed results file: file-89997952a6fe4f27a6696af0b496cca9",
      "type": "message",
      "object": "fine_tuning.job.event"
    },
    {
      "id": "ftevent-22fa0c84032940a99bbffcab783164c1",
      "created_at": 1779265623,
      "level": "info",
      "message": "Job succeeded.",
      "type": "message",
      "object": "fine_tuning.job.event"
    },
    {
      "id": "ftevent-a79d286fe43541eaa4a8446600babd34",
      "created_at": 1779264827,
      "level": "info",
      "message": "Created results file: file-89997952a6fe4f27a6696af0b496cca9",
      "type": "message",
      "object

## 6. Deploy the Fine-Tuned Model

Deployment uses the **Azure Management REST API** (control plane), which requires an Azure AD bearer token rather than an API key.

Generate a token by running in your terminal:
```bash
az login
az account get-access-token --query accessToken -o tsv
```

> **Note:** You need the **Azure AI Owner** role to create deployments.


> ⚠️ **Important:** The **subset** feature is **not supported** for fine-tuned model deployments. All inference requests will only be routed to the models present in the training data.

In [ ]:
# Get the fine-tuned model name from the completed job
fine_tuned_model = final_status.get("fine_tuned_model")
print(f"Fine-tuned model: {fine_tuned_model}")

AZURE_AD_TOKEN = subprocess.check_output("az account get-access-token --query accessToken -o tsv", shell=True).decode().strip()

if fine_tuned_model:
    print(f"Deploying model '{fine_tuned_model}' as '{DEPLOYMENT_NAME}'...")
    deploy_response = deploy_finetuned_model(
        token=AZURE_AD_TOKEN,
        subscription_id=SUBSCRIPTION_ID,
        resource_group=RESOURCE_GROUP,
        resource_name=AZURE_RESOURCE_NAME,
        deployment_name=DEPLOYMENT_NAME,
        fine_tuned_model=fine_tuned_model,
        sku_name="GlobalStandard",
        sku_capacity=DEPLOYMENT_CAPACITY, #RPM
        mode="balanced"
    )
    print("Deployment created successfully!")
    print(json.dumps(deploy_response, indent=2))
else:
    print("No fine-tuned model available. Ensure the training job succeeded.")

Fine-tuned model: model-router-2025-11-18.ft-396b292fbfb940feb07f918603f0c010
Deploying model 'model-router-2025-11-18.ft-396b292fbfb940feb07f918603f0c010' as 'model-router-demo-contosoclinic'...
Deployment created successfully!
{
  "id": "/subscriptions/72c03bf3-4e69-41af-9532-dfcdc3eefef4/resourceGroups/modelrouter-eus2-rg/providers/Microsoft.CognitiveServices/accounts/modelrouter-eus2/deployments/model-router-demo-contosoclinic",
  "type": "Microsoft.CognitiveServices/accounts/deployments",
  "name": "model-router-demo-contosoclinic",
  "sku": {
    "name": "GlobalStandard",
    "capacity": 100
  },
  "properties": {
    "model": {
      "format": "OpenAI",
      "name": "model-router-2025-11-18.ft-396b292fbfb940feb07f918603f0c010",
      "version": "1",
      "sourceAccount": "/subscriptions/72c03bf3-4e69-41af-9532-dfcdc3eefef4/resourceGroups/modelrouter-eus2-rg/providers/Microsoft.CognitiveServices/accounts/modelrouter-eus2"
    },
    "versionUpgradeOption": "NoAutoUpgrade",
    

## 7. Evaluate Fine-Tuned vs Base Model

Compare the fine-tuned model router against the base model router across test datasets (`*_test_*.jsonl` in the data folder).

For each test file, two tables are generated:
1. **Model Distribution** — which LLMs each router selects (count & percentage).
2. **Accuracy & Cost** — accuracy vs ground-truth metrics and token savings.

In [6]:
from src.evaluation_helpers import evaluate_and_compare

# Endpoint: https://<YOUR_RESOURCE_NAME>.openai.azure.com
# Both base and fine-tuned deployments use the same endpoint
EVAL_ENDPOINT = f"https://{AZURE_RESOURCE_NAME}.openai.azure.com"  # e.g. "https://myresource.openai.azure.com"
EVAL_API_KEY = AZURE_OPENAI_API_KEY
BASE_DEPLOYMENT_NAME = "model-router-demo-base"  # e.g. "model-router-demo-base"
FT_DEPLOYMENT_NAME = DEPLOYMENT_NAME  # e.g. "model-router-demo-contosoclinic-2"

evaluate_and_compare(
    endpoint=EVAL_ENDPOINT,
    api_key=EVAL_API_KEY,
    base_deployment=BASE_DEPLOYMENT_NAME,
    ft_deployment=FT_DEPLOYMENT_NAME,
    data_folder="data",
    test_pattern="*_test_*.jsonl",
)




Medical Query Testset
─────────────────────
Deployment             gpt-5-mini_2025-08-07        gpt-5_2025-08-07
--------------------------------------------------------------------
Base Model                            100.0%                    0.0%
Fine-Tuned Model                        0.0%                  100.0%


General Query Testset
─────────────────────
Deployment             gpt-5-mini_2025-08-07   gpt-5-nano_2025-08-07        gpt-5_2025-08-07
--------------------------------------------------------------------------------------------
Base Model                            100.0%                    0.0%                    0.0%
Fine-Tuned Model                       19.2%                   67.7%                   13.1%


Prod Traffic Sample Testset
───────────────────────────
Deployment               Accuracy     Actual Cost   Cost Saving vs GPT-5
------------------------------------------------------------------------
Base Model                  70.0%         $0.0068        